In [2]:
from pathlib import Path
import argparse
import sys
import cv2
from ultralytics import YOLO

DEFAULT_MODEL = Path("E:/Realtime_skin_care_analyst/skin_problem_model/best.pt")
CLASS_NAMES = ["Acne", "Dark Circle", "Pigmentation", "Spots", "Wrinkle"]

In [3]:
def load_skin_problem_model(model_path=DEFAULT_MODEL):
    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(
            f"Segmentation weights not found: {model_path}\n"
            "Copy best.pt from the Kaggle model package to this location."
        )
    return YOLO(str(model_path))


def predict_skin_problems(model, image_bgr, conf=0.25, imgsz=640):
    """
    Run segmentation on an OpenCV BGR image.

    Returns:
        annotated_bgr: image with masks/boxes/labels drawn
        summary: list of dicts with class, confidence, and bounding box
        result: original Ultralytics result (contains masks when detected)

    Pass a face crop here if that matches the model's training images.
    Pass the full frame if the dataset/model was trained on full-frame images.
    """
    if image_bgr is None or image_bgr.size == 0:
        raise ValueError("Input image is empty.")

    result = model.predict(
        source=image_bgr,
        imgsz=imgsz,
        conf=conf,
        verbose=False,
    )[0]

    # Ultralytics plots masks and boxes in the same coordinate space as input.
    annotated_bgr = result.plot(
        boxes=True,
        masks=True,
        labels=True,
        probs=False,
    )

    summary = []
    if result.boxes is not None and len(result.boxes) > 0:
        xyxy = result.boxes.xyxy.cpu().numpy()
        cls_ids = result.boxes.cls.cpu().numpy().astype(int)
        confs = result.boxes.conf.cpu().numpy()
        for box, cls_id, score in zip(xyxy, cls_ids, confs):
            name = (
                result.names.get(int(cls_id), str(cls_id))
                if isinstance(result.names, dict)
                else CLASS_NAMES[int(cls_id)]
            )
            summary.append({
                "class": name,
                "confidence": float(score),
                "bbox_xyxy": [int(v) for v in box],
            })
    return annotated_bgr, summary, result

In [4]:
def run_image(model, image_path, output_path, conf=0.25, imgsz=640):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    annotated, summary, _ = predict_skin_problems(model, image, conf=conf, imgsz=imgsz)
    cv2.imwrite(str(output_path), annotated)
    print("Predictions:")
    if summary:
        for item in summary:
            print(f"  {item['class']}: {item['confidence']:.3f} | bbox={item['bbox_xyxy']}")
    else:
        print("  No problem regions detected above the confidence threshold.")
    print(f"Saved visualization: {output_path}")


def run_webcam(model, camera_index=0, conf=0.25, imgsz=640):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open webcam index {camera_index}.")

    print("Press Q to quit.")
    last_summary = []
    frame_count = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("Could not read a webcam frame.")
                break

            # To reduce compute, infer every second frame; display the latest result.
            # Set INFER_EVERY = 1 for inference on every frame.
            INFER_EVERY = 2
            if frame_count % INFER_EVERY == 0:
                annotated, last_summary, _ = predict_skin_problems(
                    model, frame, conf=conf, imgsz=imgsz
                )
            else:
                annotated = frame.copy()

            # Show latest skin-problem detections in a panel on the same video frame.
            panel = "Skin problems:"
            if last_summary:
                lines = []
                for item in last_summary[:5]:
                    lines.append(f"{item['class']}: {item['confidence']:.2f}")
                panel += " " + " | ".join(lines)
            else:
                panel += " no regions above threshold"

            cv2.rectangle(annotated, (0, 0), (annotated.shape[1], 34), (25, 25, 25), -1)
            cv2.putText(
                annotated, panel[:120], (10, 23),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA
            )
            cv2.imshow("Skin Problem Segmentation - press Q to quit", annotated)
            frame_count += 1
            if cv2.waitKey(1) & 0xFF in (ord("q"), ord("Q")):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--model", default=str(DEFAULT_MODEL))
parser.add_argument("--source", choices=["webcam", "image"], default="webcam")
parser.add_argument("--image", help="Path to an image when --source image")
parser.add_argument("--output", default="skin_problem_prediction.jpg")
parser.add_argument("--camera", type=int, default=0)
parser.add_argument("--conf", type=float, default=0.25)
parser.add_argument("--imgsz", type=int, default=640)
# Jupyter adds its own kernel connection arguments to sys.argv.
args = parser.parse_args([] if "ipykernel" in sys.modules else None)

model = load_skin_problem_model(args.model)
if args.source == "image":
    if not args.image:
        parser.error("--image is required when --source image")
    run_image(model, args.image, args.output, args.conf, args.imgsz)
else:
    run_webcam(model, args.camera, args.conf, args.imgsz)

Press Q to quit.


In [ ]:
import cv2
import numpy as np

def draw_skin_problem_overlay(frame, model, face_crop=None, face_xyxy=None,
                              conf=0.25, imgsz=640, alpha=0.30):
    """
    Predict skin-problem masks on face_crop and draw them on frame.

    Args:
        frame: original OpenCV BGR webcam frame.
        model: loaded Ultralytics YOLO segmentation model.
        face_crop: crop passed to the segmentation model. If None, predicts
                   on the full frame.
        face_xyxy: (x1, y1, x2, y2) crop bounds in the original frame.
                   Required when face_crop is supplied.
        conf: confidence threshold.
        imgsz: inference image size.
        alpha: transparency of filled masks.

    Returns:
        (frame_with_overlay, detections)
    """
    if frame is None or frame.size == 0:
        return frame, []

    use_crop = face_crop is not None and face_xyxy is not None
    source = face_crop if use_crop else frame
    if source is None or source.size == 0:
        return frame, []

    result = model.predict(source=source, imgsz=imgsz, conf=conf, verbose=False)[0]
    output = frame.copy()
    overlay = output.copy()
    detections = []

    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return output, detections

    cls_ids = result.boxes.cls.cpu().numpy().astype(int)
    scores = result.boxes.conf.cpu().numpy()
    boxes = result.boxes.xyxy.cpu().numpy()
    polygons = result.masks.xy  # polygon coordinates in source image coordinates

    # Stable class colors in BGR order.
    colors = {
        0: (40, 210, 40),    # Acne
        1: (220, 120, 40),   # Dark Circle
        2: (40, 180, 240),   # Pigmentation
        3: (220, 60, 220),   # Spots
        4: (40, 220, 240),   # Wrinkle
    }
    names = result.names

    if use_crop:
        xoff, yoff, x2, y2 = [int(v) for v in face_xyxy]
        xoff = max(0, xoff)
        yoff = max(0, yoff)
    else:
        xoff, yoff = 0, 0

    H, W = output.shape[:2]

    for i, (cls_id, score, box) in enumerate(zip(cls_ids, scores, boxes)):
        cls_id = int(cls_id)
        color = colors.get(cls_id, (0, 255, 255))
        name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)

        if i < len(polygons):
            poly = np.asarray(polygons[i], dtype=np.float32)
            if poly.ndim == 2 and len(poly) >= 3:
                poly[:, 0] += xoff
                poly[:, 1] += yoff
                poly[:, 0] = np.clip(poly[:, 0], 0, W - 1)
                poly[:, 1] = np.clip(poly[:, 1], 0, H - 1)
                poly_i = poly.astype(np.int32).reshape((-1, 1, 2))
                cv2.fillPoly(overlay, [poly_i], color)
                cv2.polylines(output, [poly_i], True, color, 2, cv2.LINE_AA)

        bx1, by1, bx2, by2 = [int(v) for v in box]
        bx1, bx2 = bx1 + xoff, bx2 + xoff
        by1, by2 = by1 + yoff, by2 + yoff
        bx1, by1 = max(0, bx1), max(0, by1)
        bx2, by2 = min(W - 1, bx2), min(H - 1, by2)

        cv2.rectangle(output, (bx1, by1), (bx2, by2), color, 1, cv2.LINE_AA)
        label = f"{name} {float(score):.2f}"
        text_y = max(18, by1 - 6)
        cv2.putText(output, label, (bx1, text_y), cv2.FONT_HERSHEY_SIMPLEX,
                    0.48, color, 2, cv2.LINE_AA)
        detections.append({
            "class": name,
            "confidence": float(score),
            "bbox_xyxy": [bx1, by1, bx2, by2],
        })

    output = cv2.addWeighted(overlay, alpha, output, 1 - alpha, 0)
    # Redraw polygon outlines and labels after blending to keep them visible.
    for i, poly in enumerate(polygons):
        poly = np.asarray(poly, dtype=np.float32)
        if poly.ndim != 2 or len(poly) < 3:
            continue
        poly[:, 0] += xoff
        poly[:, 1] += yoff
        poly[:, 0] = np.clip(poly[:, 0], 0, W - 1)
        poly[:, 1] = np.clip(poly[:, 1], 0, H - 1)
        cv2.polylines(output, [poly.astype(np.int32).reshape((-1, 1, 2))],
                      True, colors.get(int(cls_ids[i]), (0, 255, 255)), 2, cv2.LINE_AA)